# Image Feature Extraction

In [1]:
import os
import numpy as np
import pandas as pd

In [ ]:

dataset_name = 'Beauty'
os.chdir('./datasets/Amazon-2014/%s/5-core'%dataset_name)
os.getcwd()


## Load meta_data

In [3]:
i_id, desc_str = 'itemID', 'description'

file_path = './'
file_name = 'meta-%s.csv'%dataset_name

meta_file = os.path.join(file_path, file_name)

df = pd.read_csv(meta_file)
df.sort_values(by=[i_id], inplace=True)

print('data loaded!')
print(f'shape: {df.shape}')

df[:3]

data loaded!
shape: (11924, 10)


,itemID,asin,description,title,price,salesRank,imUrl,brand,categories,related
0,0,0439893577,The Magnetic Tabletop Learning Easel is one of...,Little Red Tool Box: Magnetic Tabletop Learnin...,16.19,NaN,http://ecx.images-amazon.com/images/I/51XCjcMt...,Scholastic,"[['Toys & Games', 'Dress Up & Pretend Play', '...","{'also_bought': ['B0015KEB2M', 'B001ECI9EM', '..."
1,1,048645195X,NaN,Dover Publications-Decorative Tile Designs Col...,1.44,"{'Arts, Crafts & Sewing': 1009}",http://ecx.images-amazon.com/images/I/61r4Gp10...,Dover Pubns,"[['Toys & Games', 'Arts & Crafts', 'Drawing & ...","{'also_bought': ['0486456943', '0486456420', '..."
2,2,0545496470,NaN,The Book of Impossible Objects: 25 Eye-Popping...,11.13,"{'Arts, Crafts & Sewing': 1462}",http://ecx.images-amazon.com/images/I/61mEDfXS...,Klutz,"[['Toys & Games', 'Novelty & Gag Toys', 'Magic...","{'also_bought': ['1570548943', '1452109036', '..."


# Image Feature Exatrct

In [ ]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
from tqdm import tqdm

gpu_id = 1
device = torch.device('cuda:' + str(gpu_id) if torch.cuda.is_available() else 'cpu')

model_name = "./pretrain_models/clip-vit-base-patch32" 
processor = CLIPProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name)
model = model.to(device)

def extract_features(image_path, model, processor):
    """ Use CLIP extract image feature """
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt")
    inputs = inputs.to(model.device)
    with torch.no_grad():
        image_features =  model.get_image_features(**inputs)
    return image_features

image_folder = "../images"
asin_list = df['asin'].astype(str).tolist()
itemid_list = df['itemID'].tolist()

features = []
item_ids = [] # all items that have image
batch_size = 32
image_files = [f for f in os.listdir(image_folder) if f.endswith(".jpg")]
for i in tqdm(range(0, len(image_files), batch_size), desc="Processing Batches"):
    batch_asins = asin_list[i:i + batch_size]
    batch_itemids = itemid_list[i:i + batch_size]

    for asin, item_id in zip(batch_asins, batch_itemids):
        image_path = os.path.join(image_folder, f"{asin}.jpg")
        if not os.path.exists(image_path):
            print(f"[skip] Picture missing: {asin}.jpg")
            continue
        image_feature = extract_features(image_path, model, processor)

        image_feature = image_feature.detach().cpu().numpy()
        if len(features) == 0:
            features = image_feature
        else:
            features = np.concatenate((features, image_feature), axis=0)
        item_ids.append(item_id)

print(features.shape)
features[:5]

In [ ]:
average_feature = np.mean(features, axis=0)

# check null
features_array = np.empty((df.shape[0], features.shape[-1]))
idx = 0
for item_id in df['itemID'].values:
    if item_id not in item_ids:
        features_array[item_id] = average_feature
    else:
        features_array[item_id] = features[idx]
        idx += 1

assert features_array.shape[0] == df.shape[0]

print(features_array.shape)
features_array[:5]

(11924, 512)


array([[-0.07346472, -0.18123916,  0.35176322, ...,  0.1225111 ,
        -0.85880888,  0.09946784],
       [-0.11662436,  0.25708342,  0.15292399, ...,  0.06471419,
        -0.47544301,  0.07605848],
       [ 0.14538945,  0.0541929 ,  0.1355608 , ...,  0.06097623,
        -0.22840551,  0.23211607],
       [ 0.45850825, -0.26763099,  0.65284896, ..., -0.01097409,
        -0.27687228,  0.10084938],
       [ 0.16031358,  0.14894983,  0.37965441, ...,  0.01493033,
        -0.03188802, -0.10513278]])

In [6]:
# save
np.save(os.path.join(file_path, 'image_feat.npy'), features_array)
print('done!')

done!


In [7]:
# load
load_img_feat = np.load('image_feat.npy', allow_pickle=True)
print(load_img_feat.shape)
load_img_feat[:10]

(11924, 512)


array([[-0.07346472, -0.18123916,  0.35176322, ...,  0.1225111 ,
        -0.85880888,  0.09946784],
       [-0.11662436,  0.25708342,  0.15292399, ...,  0.06471419,
        -0.47544301,  0.07605848],
       [ 0.14538945,  0.0541929 ,  0.1355608 , ...,  0.06097623,
        -0.22840551,  0.23211607],
       ...,
       [ 0.15470159, -0.46080941,  0.04973588, ...,  0.20954773,
        -0.37799066,  0.26751992],
       [-0.29094344,  0.25102681,  0.23508522, ...,  0.35368487,
        -0.27378356, -0.0143871 ],
       [-0.26462919, -0.05072594,  0.49692491, ...,  0.45152929,
        -0.26486281,  0.02729651]])